# MAMMAL Model Fine-Tuning

Fine-tune a pretrained MAMMAL model on a **binary ligand-target classification** task.

### Model prompt format

The encoder input uses a **generic SMILES prompt** (no task-specific token in the prompt):
```
<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0><MOLECULAR_ENTITY><MOLECULAR_ENTITY_SMALL_MOLECULE>
<@TOKENIZER-TYPE=SMILES@MAX-LEN=300><SEQUENCE_NATURAL_START>{smiles}<SEQUENCE_NATURAL_END><EOS>
```
`target_name` and `assay_type` are used only for naming the task and the validation metric key.

## Steps
0. Configuration — edit paths and hyperparameters here
1. Preview training data
2. Load tokenizer
3. Load pretrained MAMMAL model
4. Initialise task & data module
5. Build Lightning module & trainer
6. Run fine-tuning
7. Inspect training results

## 0. Configuration

**Edit the settings below before running.**

In [2]:
def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False
IN_COLAB = in_colab()
print(f"{IN_COLAB=}")   

IN_COLAB=False


In [ ]:
import os
import torch
run_on_wdr91 = True

if IN_COLAB:
    MODELS_DIR = "/Content/models"
else:
    USER = os.getenv("USER")
    MODELS_DIR = f"/proj/ligand_ai/{USER}"

if run_on_wdr91:
    NAME = "wdr91_asms"
    if IN_COLAB:
        DATA_PATH = "/Content/data"
    else:
        DATA_PATH = "/proj/ligand_ai/datasets_manager/processed/splits/wdr91_ASMS_train_val_v1"
else:
    NAME = "pgk2_del_cdd"
    if IN_COLAB:
        DATA_PATH = "/Content/data"
    else:
        DATA_PATH = "/proj/ligand_ai/datasets_manager/processed/splits/PGK2_DEL_cdd_to_creative"

MODEL_DIR = os.path.join(MODELS_DIR, NAME)
# Column names
SMILES_COLUMN = "smiles"
LABEL_COLUMN = "label"

# Base MAMMAL model (HuggingFace hub ID or local path)
BASE_MODEL_PATH = "ibm/biomed.omics.bl.sm.ma-ted-458m"

assert os.path.exists(DATA_PATH),  f"Data not found:  {DATA_PATH}"
os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)

TENSORBOARD_LOG_DIR = os.path.join(MODEL_DIR, "tensorboard_logs")

# Inference settings
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"{MODEL_DIR=}")
print(f"{DATA_PATH=}")
print(f"{DEVICE=}")

MODEL_DIR='/proj/ligand_ai/ozery/wdr91_asms'
DATA_PATH='/proj/ligand_ai/datasets_manager/processed/splits/wdr91_ASMS_train_val_v1'
DEVICE='cuda'


In [ ]:
# import os
# import torch

# # ── USER SETTINGS ──────────────────────────────────────────────────────────────

# # Experiment name
# NAME = "wdr91_ASMS_train_val_v1_mammal"

# # Directory where the fine-tuned model checkpoints will be saved
# MODEL_DIR = "/proj/ligand_ai/models/wdr91_ASMS_train_val_v1/mammal_fixed"

# # Target protein name and assay type
# # Used for: task name (e.g. 'wdr91_asms') and validation metric key
# # NOT used in the model prompt (generic SMILES prompt is used instead)
# TARGET_NAME = "WDR91"
# ASSAY_TYPE  = "ASMS"

# # Path to the dataset splits directory
# # Expected to contain train.csv, val.csv (and optionally test.csv)
# DATA_PATH = "/proj/ligand_ai/datasets_manager/processed/splits/wdr91_ASMS_train_val_v1"

# # Column names in the CSV files
# SMILES_COLUMN = "smiles"
# LABEL_COLUMN  = "label"

# # Base MAMMAL model (HuggingFace hub ID or local path)
# BASE_MODEL_PATH = "ibm/biomed.omics.bl.sm.ma-ted-458m"

# # Training hyperparameters
# MAX_EPOCHS            = 10
# BATCH_SIZE            = 128
# LEARNING_RATE         = 1e-5
# DRUG_MAX_SEQ_LENGTH   = 300
# ENCODER_INPUT_MAX_LEN = 320
# LABELS_MAX_LEN        = 4
# NUM_WORKERS           = 0   # set > 0 for faster data loading on multi-core machines
# SEED                  = 1234

# # LR scheduler (cosine annealing with warmup)
# LR_T_MAX          = 1_000_000
# LR_WARMUP_STEPS   = 2_000
# LR_ETA_MIN_FACTOR = 0.1

# # ClearML experiment tracking
# # Set CLEARML_PROJECT to None to disable tracking
# CLEARML_PROJECT = "sgc-ligand-ai/mammal"
# CLEARML_TASK    = "/proj/ligand_ai/models/wdr91_ASMS_train_val_v1_mammal"
# CLEARML_TAGS    = ["wdr91_ASMS", "train_val_v1", "mammal"]

# # Device
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# # ── VALIDATION ─────────────────────────────────────────────────────────────────
# assert os.path.isdir(DATA_PATH), f"Data directory not found: {DATA_PATH}"
# os.makedirs(MODEL_DIR, exist_ok=True)

# print(f"Name      : {NAME}")
# print(f"Target    : {TARGET_NAME} / {ASSAY_TYPE}")
# print(f"Data      : {DATA_PATH}")
# print(f"Model dir : {MODEL_DIR}")
# print(f"Device    : {DEVICE}")
# print(f"Max epochs: {MAX_EPOCHS}")

Name      : wdr91_ASMS_train_val_v1_mammal
Target    : WDR91 / ASMS
Data      : /proj/ligand_ai/datasets_manager/processed/splits/wdr91_ASMS_train_val_v1
Model dir : /proj/ligand_ai/models/wdr91_ASMS_train_val_v1/mammal_fixed
Device    : cpu
Max epochs: 10


## 1. Preview Training Data

In [6]:
import pandas as pd

splits = {}
for split in ("train", "val"):
    print(f"***{split=}")
    path = os.path.join(DATA_PATH, f"{split}.csv")    
    splits[split] = pd.read_csv(path)
    
    display(splits[split])
    print(f"\nLabel distribution (train):")
    print(splits[split][LABEL_COLUMN].value_counts().sort_index())
    
    

***split='train'


,smiles,label
0,O=C(COc1ccccc1)NC(c1ccccc1)c1ccccc1,0
1,CCC1(CO)CCCN(c2nccn3nc(C)cc23)C1,0
2,CCCN(C(=O)c1cc(-c2c(C)nn(C)c2C)n[nH]1)C1CCc2cc...,0
3,COCCc1nc(CN2C[C@H](OC)C[C@H]2c2nc(C)c(C)[nH]2)no1,0
4,COc1ccc(CC(=O)Nc2sccc2C#N)cc1F,0
...,...,...
305327,CN(CC(=O)Nc1ccccc1Br)C(=O)CN(C)C1CCCCC1,0
305328,Cc1ccn2c(=O)cc(Cn3nc(C)c(C)c(C#N)c3=O)nc2c1,0
305329,COc1cc(OC)c(S(=O)(=O)N2CCC(N)CC2)cc1Br,0
305330,O=C(Nc1cccc(OC2CCCC2)c1)C1COc2ccccc2O1,0



Label distribution (train):
label
0    305208
1       124
Name: count, dtype: int64
***split='val'


,smiles,label
0,Fc1ccccc1CN(c1ccc2nnc(C(F)(F)F)n2n1)C1CC1,0
1,Cc1cc(NC(=O)N2CC[C@H]3[C@@H](C2)NC(=O)N3C(C)C)...,0
2,COc1coc(C(=O)Nc2cc(C3CCCOC3)[nH]n2)cc1=O,0
3,O=C(NCC1(c2c(F)cccc2F)CCCC1)c1ccc(=O)[nH]n1,0
4,Cc1cc(C(=O)N2CCOC(c3ccc(F)c(Cl)c3)C2)no1,0
...,...,...
33921,Cc1cc(NC(=O)CN2C(C(=O)O)CC3CCCCC32)no1,0
33922,O=C(O)C1CCCN1C(=O)c1cnc2ccccn2c1=O,0
33923,CN1CCN(CCCNC(=O)Nc2ccc(C(C)(C)C)cc2)CC1,0
33924,Cc1cnc2c(S(=O)(=O)N[C@H]3C[C@@](C)(O)C3)cccc2c1,0



Label distribution (train):
label
0    33912
1       14
Name: count, dtype: int64


## 2. Load Tokenizer

In [7]:
from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp

tokenizer_op = ModularTokenizerOp.from_pretrained(BASE_MODEL_PATH)
print("Tokenizer loaded.")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

The OrderedVocab you are attempting to save contains holes for indices [314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499

34, 4935, 4936, 4937, 4938, 4939, 4940, 4941, 4942, 4943, 4944, 4945, 4946, 4947, 4948, 4949, 4950, 4951, 4952, 4953, 4954, 4955, 4956, 4957, 4958, 4959, 4960, 4961, 4962, 4963, 4964, 4965, 4966, 4967, 4968, 4969, 4970, 4971, 4972, 4973, 4974, 4975, 4976, 4977, 4978, 4979, 4980, 4981, 4982, 4983, 4984, 4985, 4986, 4987, 4988, 4989, 4990, 4991, 4992, 4993, 4994, 4995, 4996, 4997, 4998, 4999], your vocabulary could be corrupted !


## 3. Load Pretrained MAMMAL Model

In [8]:
from mammal.model import Mammal

model = Mammal.from_pretrained(BASE_MODEL_PATH)
print(model)

2026-03-04 21:55:47.635455: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/proj/bmfm/users/ozery/miniforge3/envs/fuse4/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/proj/bmfm/users/ozery/miniforge3/envs/fuse4/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compati

Path doesn't exist. Will try to download from hf hub. pretrained_model_name_or_path='ibm/biomed.omics.bl.sm.ma-ted-458m'


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Attempting to load model from dir: pretrained_model_name_or_path='/u/ozery/.cache/huggingface/hub/models--ibm--biomed.omics.bl.sm.ma-ted-458m/snapshots/6d319d8dcf97f8821635327fc8cda24670553daa'
Mammal(
  (t5_model): T5ForConditionalGeneration(
    (shared): Embedding(100001, 768)
    (encoder): T5Stack(
      (embed_tokens): Embedding(100001, 768)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=768, out_features=768, bias=False)
                (k): Linear(in_features=768, out_features=768, bias=False)
                (v): Linear(in_features=768, out_features=768, bias=False)
                (o): Linear(in_features=768, out_features=768, bias=False)
                (relative_attention_bias): Embedding(32, 12)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )

## 4. Initialise Task & Data Module

In [10]:
from torch.utils.data import Dataset

class PreprocessedDataset(Dataset):
    
    def __init__(self, base_dataset: Dataset, preprocessing_fn: callable, preprocessing_kwargs: dict):
        self.base_dataset = base_dataset
        self.preprocessing_fn = preprocessing_fn
        self.preprocessing_kwargs = preprocessing_kwargs
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        sample = self.base_dataset[idx]
        processed_sample = self.preprocessing_fn(sample, **self.preprocessing_kwargs)
        return processed_sample

In [ ]:
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset


class SMILESDataset(Dataset):

    def __init__(
        self,
        data_path: str,
        set_name: str,
        smiles_column: str = "smiles",
        label_column: str = "label",
    ) -> None:
        """Initialize dataset by loading data from CSV/TSV file."""
        self.data_path = Path(data_path)
        self.set_name = set_name
        self.smiles_column = smiles_column
        self.label_column = label_column
        
        # Load data file
        df = self._load_dataframe()
        
    
        # Convert to lists for fast access (avoids DataFrame indexing overhead)
        self.sample_ids = df["smiles_column"].tolist()
        self.smiles = df[smiles_column].tolist()
        self.labels = df[label_column].tolist()
        
        print(f"FastSMILESDataset: Loaded {set_name} set with {len(self)} samples")
    
    def _load_dataframe(self) -> pd.DataFrame:
        file_path = self.data_path / f"{self.set_name}.csv"
        return pd.read_csv(file_path)
        
    
    def __len__(self) -> int:
        return len(self.smiles)
    
    def __getitem__(self, idx: int) -> dict:
        
        return {
            "data.sample_id": self.sample_ids[idx],
            "data.smiles": self.smiles[idx],
            "data.label": self.labels[idx],
        }


In [13]:
import pytorch_lightning as pl
from functools import partial
from mammal.keys import *
from torch.utils.data.dataloader import DataLoader
from fuse.data.utils.collates import CollateDefault


class LigandTargetDataModule(pl.LightningDataModule):
    def __init__(
        self,
        *,
        task_name: str,
        data_path: str,
        batch_size: int,
        tokenizer_op: ModularTokenizerOp,
        drug_max_seq_length: int,
        encoder_input_max_seq_len: int,
        data_preprocessing: callable,
        labels_max_seq_len: int,
        smiles_column: str = "smiles",
        label_column: str = "label",
        num_workers: int = 0,
        pin_memory: bool = True,
        persistent_workers: bool = False,
    ) -> None:
        super().__init__()
        self.tokenizer_op = tokenizer_op
        self.drug_max_seq_length = drug_max_seq_length
        self.encoder_input_max_seq_len = encoder_input_max_seq_len
        self.labels_max_seq_len = labels_max_seq_len
        self.batch_size = batch_size
        self.data_preprocessing = data_preprocessing
        self.pad_token_id = self.tokenizer_op.get_token_id("<PAD>")
        
        self.task_name = task_name
        self.data_path = data_path
        self.smiles_column = smiles_column
        self.label_column = label_column
        
        # Performance parameters
        self.num_workers = num_workers
        self.pin_memory = pin_memory
        self.persistent_workers = persistent_workers if num_workers > 0 else False
        
        # Setup padding crop optimization (10-20% speedup)
        # This removes unnecessary padding tokens before passing to model
        self.special_handlers = {
            ENCODER_INPUTS_TOKENS: partial(
                CollateDefault.crop_padding, pad_token_id=self.pad_token_id
            ),
            ENCODER_INPUTS_ATTENTION_MASK: partial(
                CollateDefault.crop_padding, pad_token_id=False
            ),
        }

    def setup(self, stage: str) -> None:
        """Setup datasets for training, validation, and testing."""
        # Load raw datasets
        raw_ds_dict = self.load_datasets()
        
        # Preprocessing kwargs
        preprocessing_kwargs = dict(
            sequence_key="data.smiles",
            label_key="data.label",
            target_name=self.target_name,
            assay_type=self.assay_type,
            tokenizer_op=self.tokenizer_op,
            drug_max_seq_length=self.drug_max_seq_length,
            encoder_input_max_seq_len=self.encoder_input_max_seq_len,
            labels_max_seq_len=self.labels_max_seq_len,
        )
        
        # Wrap each dataset with preprocessing
        self.ds_dict = {}
        for name, raw_ds in raw_ds_dict.items():
            self.ds_dict[name] = PreprocessedDataset(
                base_dataset=raw_ds,
                preprocessing_fn=self.data_preprocessing,
                preprocessing_kwargs=preprocessing_kwargs,
            )

    def load_datasets(self) -> dict[str, SMILESDataset]:
        ds_dict = {}
        
        for set_name in ["train", "val", "test"]:
            # Create fast PyTorch dataset - directly usable with DataLoader
            pytorch_dataset = SMILESDataset(
                data_path=self.data_path,
                set_name=set_name,
                smiles_column=self.smiles_column,
                label_column=self.label_column,
            )
            
            # Map 'val' to 'valid' for consistency
            ds_name = "valid" if set_name == "val" else set_name
            ds_dict[ds_name] = pytorch_dataset

        return ds_dict

    def train_dataloader(self) -> DataLoader:
        """Create training data loader with padding crop optimization."""
        train_loader = DataLoader(
            dataset=self.ds_dict["train"],
            batch_size=self.batch_size,
            collate_fn=CollateDefault(special_handlers_keys=self.special_handlers),
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            persistent_workers=self.persistent_workers,
        )
        return train_loader

    def val_dataloader(self) -> DataLoader:
        """Create validation data loader with padding crop optimization."""
        val_loader = DataLoader(
            self.ds_dict["valid"],
            batch_size=self.batch_size,
            collate_fn=CollateDefault(special_handlers_keys=self.special_handlers),
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            persistent_workers=self.persistent_workers,
        )
        return val_loader

    def test_dataloader(self) -> DataLoader:
        """Create test data loader with padding crop optimization."""
        test_loader = DataLoader(
            self.ds_dict["test"],
            batch_size=self.batch_size,
            collate_fn=CollateDefault(special_handlers_keys=self.special_handlers),
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            persistent_workers=self.persistent_workers,
        )
        return test_loader

    def predict_dataloader(self) -> DataLoader:
        """Create prediction data loader (uses test set)."""
        return self.test_dataloader()

In [ ]:
from mammal.metrics import classification_metrics
from mammal.task import (
    MammalTask,
    MetricBase,
)
import pytorch_lightning as pl

from mammal.keys import *
import numpy as np
from tqdm import tqdm

class LigandTargetTask(MammalTask):
    def __init__(
        self,
        *,
        task_name: str,
        tokenizer_op: ModularTokenizerOp,
        data_module_kwargs: dict,
        logger = None,
    ) -> None:

        super().__init__(
            name=task_name,
            logger=logger,
            tokenizer_op=tokenizer_op,
        )
        self._data_module_kwargs = data_module_kwargs
        self.task_name = task_name


    def data_module(self) -> pl.LightningDataModule:
        """Create and return the data module."""
        from .pl_data_module import LigandTargetDataModule
        
        return LigandTargetDataModule(
            tokenizer_op=self._tokenizer_op,
            data_preprocessing=self.data_preprocessing,
            task_name=self.task_name,
            **self._data_module_kwargs,
        )

    def train_metrics(self) -> dict[str, MetricBase]:
        metrics = super().train_metrics()
        
        metrics.update(self.get_classification_metrics())
        return metrics

    def validation_metrics(self) -> dict[str, MetricBase]:
        """Define validation metrics with safe error handling."""
        validation_metrics = super().validation_metrics()
        
        validation_metrics.update(self.get_classification_metrics())
        return validation_metrics
    
    def get_classification_metrics(self) -> dict[str, MetricBase]:
        return classification_metrics(
            self.name(),
            class_position=1,
            tokenizer_op=self._tokenizer_op,
            class_tokens=["<0>", "<1>"],
        )

    @staticmethod
    def data_preprocessing(
        sample_dict: dict,
        *,
        sequence_key: str,
        label_key: str | None = None,
        drug_max_seq_length: int = 1250,
        encoder_input_max_seq_len: int | None = 1260,
        labels_max_seq_len: int | None = 4,
        tokenizer_op: ModularTokenizerOp,
        device: str | torch.device = "cpu",
    ) -> dict:

        drug_sequence = sample_dict[sequence_key]
        label = sample_dict.get(label_key, None) if label_key else None
    
        sample_dict[ENCODER_INPUTS_STR] = (
            f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0>"
            f"<MOLECULAR_ENTITY><MOLECULAR_ENTITY_SMALL_MOLECULE>"
            f"<@TOKENIZER-TYPE=SMILES@MAX-LEN={drug_max_seq_length}>"
            f"<SEQUENCE_NATURAL_START>{drug_sequence}<SEQUENCE_NATURAL_END><EOS>"
        )
        
        # Tokenize encoder input
        tokenizer_op(
            sample_dict=sample_dict,
            key_in=ENCODER_INPUTS_STR,
            key_out_tokens_ids=ENCODER_INPUTS_TOKENS,
            key_out_attention_mask=ENCODER_INPUTS_ATTENTION_MASK,
            max_seq_len=encoder_input_max_seq_len,
        )
        sample_dict[ENCODER_INPUTS_TOKENS] = torch.tensor(
            sample_dict[ENCODER_INPUTS_TOKENS], device=device
        )
        sample_dict[ENCODER_INPUTS_ATTENTION_MASK] = torch.tensor(
            sample_dict[ENCODER_INPUTS_ATTENTION_MASK], device=device
        )

        # Process labels if available (for training/validation)
        if label is not None:
            pad_id = tokenizer_op.get_token_id("<PAD>")
            ignore_token_value = -100
            
            # Format labels
            sample_dict[LABELS_STR] = (
                f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0><{label}><EOS>"
            )
            tokenizer_op(
                sample_dict=sample_dict,
                key_in=LABELS_STR,
                key_out_tokens_ids=LABELS_TOKENS,
                key_out_attention_mask=LABELS_ATTENTION_MASK,
                max_seq_len=labels_max_seq_len,
            )
            sample_dict[LABELS_TOKENS] = torch.tensor(
                sample_dict[LABELS_TOKENS], device=device
            )
            sample_dict[LABELS_ATTENTION_MASK] = torch.tensor(
                sample_dict[LABELS_ATTENTION_MASK], device=device
            )
            
            # Replace pad tokens with ignore value
            sample_dict[LABELS_TOKENS][
                (sample_dict[LABELS_TOKENS][..., None] == torch.tensor(pad_id))
                .any(-1)
                .nonzero()
            ] = ignore_token_value

            # Format decoder input
            sample_dict[DECODER_INPUTS_STR] = (
                f"<@TOKENIZER-TYPE=SMILES><DECODER_START><SENTINEL_ID_0><{label}><EOS>"
            )
            tokenizer_op(
                sample_dict=sample_dict,
                key_in=DECODER_INPUTS_STR,
                key_out_tokens_ids=DECODER_INPUTS_TOKENS,
                key_out_attention_mask=DECODER_INPUTS_ATTENTION_MASK,
                max_seq_len=labels_max_seq_len,
            )
            sample_dict[DECODER_INPUTS_TOKENS] = torch.tensor(
                sample_dict[DECODER_INPUTS_TOKENS], device=device
            )
            sample_dict[DECODER_INPUTS_ATTENTION_MASK] = torch.tensor(
                sample_dict[DECODER_INPUTS_ATTENTION_MASK], device=device
            )

        return sample_dict

    @staticmethod
    def process_model_output(
        tokenizer_op: ModularTokenizerOp,
        decoder_output: np.ndarray,
        decoder_output_scores: np.ndarray,
    ) -> dict:
        negative_token_id = tokenizer_op.get_token_id("<0>")
        positive_token_id = tokenizer_op.get_token_id("<1>")
        
        label_id_to_int = {
            negative_token_id: 0,
            positive_token_id: 1,
        }
        classification_position = 1

        # Get the predicted token
        predicted_token = int(decoder_output[classification_position])

        # @TODO: scores are not normalized!!!
        if decoder_output_scores is not None:
            neg_score = decoder_output_scores[classification_position, negative_token_id]
            pos_score = decoder_output_scores[classification_position, positive_token_id]
            score = pos_score / (pos_score + neg_score + 1e-8)  # probability of positive class
        else:
            score = None

        # Handle prediction with strict/non-strict mode
        if predicted_token in label_id_to_int:
            # Token is valid (<0> or <1>)
            pred = label_id_to_int[predicted_token]
        else:
            if score is not None:
                pred = 1 if score > 0.5 else 0
            else:
                # No scores available, default to 0
                pred = None
                
        ans = dict(
            pred=pred,
            score=score,
        )

        return ans

    def predict(
        self,
        model,
        dataloader,
        device,
        return_dataframe: bool = True,
    ) -> pd.DataFrame | dict:
       
        model.eval()
        model = model.to(device)
        
        results = {
            "sample_id": [],
            "predicted_label": [],
            "prediction_score": [],
        }
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(tqdm(dataloader, desc="Generating predictions")):
                # Get batch size
                batch_size = batch[ENCODER_INPUTS_TOKENS].shape[0] if ENCODER_INPUTS_TOKENS in batch else len(batch.get("index", []))
                
                # Convert batch tensors to list of sample_dicts
                sample_dicts = []
                for i in range(batch_size):
                    sample_dict = {}
                    for key, value in batch.items():
                        if isinstance(value, torch.Tensor):
                            # Slice to get individual sample, keep as 1D or 2D tensor
                            sample_dict[key] = value[i].to(device)
                        elif isinstance(value, list):
                            sample_dict[key] = value[i]
                        else:
                            sample_dict[key] = value
                    sample_dicts.append(sample_dict)
                
                
                # Generate predictions using model.generate()
                batch_dict = model.generate(
                    sample_dicts,
                    output_scores=True,
                    return_dict_in_generate=True,
                    max_new_tokens=5,
                )
                
                # Extract decoder outputs
                decoder_output = batch_dict.get(CLS_PRED, None)
                decoder_output_scores = batch_dict.get(SCORES, None)
                              
                # Process each sample in the batch
                for i in range(batch_size):
                    # Get sample info from original batch
                    sample_id = batch["data.sample_id"][i]
                    if isinstance(sample_id, torch.Tensor):
                        sample_id = sample_id.item()
                    
                    smiles = batch.get("data.smiles", [None] * batch_size)[i]
                    
                    # Get true label if available
                    true_label = batch.get("data.label", [None] * batch_size)[i]
                    if isinstance(true_label, torch.Tensor):
                        true_label = true_label.item()
                    
                    # Process model output for this sample
                    if decoder_output is not None and decoder_output_scores is not None:
                        sample_decoder_output = decoder_output[i].cpu().numpy()
                        sample_decoder_scores = decoder_output_scores[i].cpu().numpy()
                        
                        # Use the process_model_output method (aligned with molnet)
                        # Enable verbose for first few samples to debug
                        verbose = (batch_idx == 0 and i < 3)
                        processed = self.process_model_output(
                            self._tokenizer_op,
                            sample_decoder_output,
                            sample_decoder_scores,
                            verbose=verbose,
                            strict=False,  # Non-strict mode: use score-based fallback for unexpected tokens
                        )
                        
                        predicted_label = processed["pred"]
                        prediction_score = processed["score"]
                        
                    else:
                        # Fallback if output format is different
                        predicted_label = None
                        prediction_score = None

                    
                    # Store results
                    results["sample_id"].append(sample_id)
                    results["predicted_label"].append(predicted_label)
                    results["prediction_score"].append(prediction_score)
                   
        
        if return_dataframe:
            return pd.DataFrame(results)
        else:
            return results


/proj/bmfm/users/ozery/miniforge3/envs/fuse4/lib/python3.11/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [ ]:
import pytorch_lightning as pl
from ligand_ai.models.mammal_finetune.task import LigandTargetTask

# Set random seed for reproducibility
pl.seed_everything(seed=1234, workers=True)

# Optional: connect to ClearML
tensorboard_logger = None
try:
    from pytorch_lightning.loggers import TensorBoardLogger
    os.makedirs(TENSORBOARD_LOG_DIR, exist_ok=True)
    tensorboard_logger = TensorBoardLogger(
        save_dir=TENSORBOARD_LOG_DIR,
        name=NAME,
        version=None,  # auto-increment version
    )
    print(f"TensorBoard logging enabled: {TENSORBOARD_LOG_DIR}")
    print(f"To view logs, run: tensorboard --logdir={TENSORBOARD_LOG_DIR}")
except Exception as e:
    print(f"Warning: TensorBoard logging failed: {e}")
    print("Continuing without tracking...")

# Instantiate the task
# target_name and assay_type are used for naming only (not in the model prompt)
num_workers=0
task = LigandTargetTask(
    task_name=NAME,
    tokenizer_op=tokenizer_op,
    logger=None,  # TensorBoard logger will be passed to trainer instead
    data_module_kwargs=dict(
        data_path=DATA_PATH,
        smiles_column=SMILES_COLUMN,
        label_column=LABEL_COLUMN,
        batch_size=128,
        drug_max_seq_length=300,
        encoder_input_max_seq_len=320,
        labels_max_seq_len=4,
        num_workers=num_workers,
        pin_memory=(DEVICE == "cuda"),
        persistent_workers=(num_workers > 0),
    ),
)

print(f"Task name : {task.name()}")

# Build the data module
pl_data_module = task.data_module()
print("Data module ready.")

Global seed set to 1234


TensorBoard logging enabled: /proj/ligand_ai/ozery/wdr91_asms/tensorboard_logs
To view logs, run: tensorboard --logdir=/proj/ligand_ai/ozery/wdr91_asms/tensorboard_logs
Task name : wdr91_asms
Data module ready.


## 5. Build Lightning Module & Trainer

In [17]:
MAX_EPOCHS = 1
LEARNING_RATE = 1e-5
from functools import partial

import torch
from fuse.dl.lightning.pl_module import LightningModuleDefault
from mammal.lr_schedulers import cosine_annealing_with_warmup_lr_scheduler


def _configure_optimizers(module, opt_callable, lr_sch_callable):
    """Configure optimizer and LR scheduler for the Lightning module."""
    opt    = opt_callable(module.trainer.model.parameters())
    lr_sch = lr_sch_callable(opt)
    return {
        "optimizer": opt,
        "lr_scheduler": {"scheduler": lr_sch, "interval": "step"},
    }


# Optimizer factory
opt_callable = partial(torch.optim.AdamW, lr=1e-5)

# LR scheduler factory
lr_sch_callable = partial(
    cosine_annealing_with_warmup_lr_scheduler,
    T_max=1_000_000,
    num_warmup_steps=2_000,
    eta_min_factor=0.1,
)

optimizers_and_lr_schs_callable = partial(
    _configure_optimizers,
    opt_callable=opt_callable,
    lr_sch_callable=lr_sch_callable,
)

# Validation metric to monitor for best-epoch checkpointing
# monitor_metric = f"validation.metrics.auroc" @TODO: check

# Lightning module
pl_module = LightningModuleDefault(
    model=model,
    losses=task.losses(),
    validation_metrics=task.validation_metrics(),
    train_metrics=task.train_metrics(),
    optimizers_and_lr_schs=optimizers_and_lr_schs_callable,
    model_dir=MODEL_DIR,
    # best_epoch_source=dict(
    #     monitor=monitor_metric,
    #     mode="max",
    # ),
)

# Trainer
pl_trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    default_root_dir=MODEL_DIR,
    accelerator="gpu" if DEVICE == "cuda" else "cpu",
    devices=1,
    num_nodes=1,
    num_sanity_val_steps=0,
    enable_progress_bar=True,
    enable_model_summary=True,
    log_every_n_steps=10,
)

print("Lightning module and trainer ready.")
# print(f"Monitoring metric : {monitor_metric}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Lightning module and trainer ready.


## 6. Save Model Config & Tokenizer, then Run Fine-Tuning

The model config and tokenizer are saved to `MODEL_DIR` before training starts
so that the directory is self-contained for later inference.

In [18]:
import os

# ── Save model config and tokenizer ───────────────────────────────────────────
os.makedirs(MODEL_DIR, exist_ok=True)

# Save model architecture config (weights are saved by Lightning callbacks)
model._save_pretrained(MODEL_DIR, save_config_only=True)
print(f"Model config saved to: {MODEL_DIR}")

# Save tokenizer
tokenizer_save_path = os.path.join(MODEL_DIR, "tokenizer")
tokenizer_op.save_pretrained(tokenizer_save_path)
print(f"Tokenizer saved to  : {tokenizer_save_path}")

# ── Start training ─────────────────────────────────────────────────────────────
print()
print("=" * 80)
print("STARTING FINE-TUNING")
print("=" * 80)
print(f"  Data      : {DATA_PATH}")
print(f"  Model dir : {MODEL_DIR}")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  LR        : {LEARNING_RATE}")
print(f"  Device    : {DEVICE}")
print("=" * 80)
print()

pl_trainer.fit(model=pl_module, datamodule=pl_data_module)

print()
print("=" * 80)
print("FINE-TUNING COMPLETED")
print("=" * 80)
print(f"Checkpoints saved to: {MODEL_DIR}")

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
Missing logger folder: /proj/ligand_ai/ozery/wdr91_asms/lightning_logs


Saving @ /proj/ligand_ai/ozery/wdr91_asms
Model config saved to: /proj/ligand_ai/ozery/wdr91_asms
Saving @ save_directory='/proj/ligand_ai/ozery/wdr91_asms/tokenizer'
The OrderedVocab you are attempting to save contains holes for indices [314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 4

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name   | Type   | Params
----------------------------------
0 | _model | Mammal | 458 M 
----------------------------------
458 M     Trainable params
0         Non-trainable params
458 M     Total params
1,832.029 Total estimated model params size (MB)
/proj/bmfm/users/ozery/miniforge3/envs/fuse4/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:224: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 256 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(
/proj/bmfm/users/ozery/miniforge3/envs/fuse4/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:224: PossibleUserWarning: The dataloader, val_dataloader 0, does not have many workers which may be a bottleneck. Consider increasing the value o

Training: 0it [00:00, ?it/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.



FINE-TUNING COMPLETED
Checkpoints saved to: /proj/ligand_ai/ozery/wdr91_asms


## 7. Inspect Training Results

List the saved checkpoints and show the logged metrics.

In [19]:
import glob

# ── List saved checkpoints ─────────────────────────────────────────────────────
ckpt_pattern = os.path.join(MODEL_DIR, "**", "*.ckpt")
checkpoints  = sorted(glob.glob(ckpt_pattern, recursive=True))

print(f"Checkpoints found in {MODEL_DIR}:")
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f"  {os.path.relpath(ckpt, MODEL_DIR):<60s}  {size_mb:6.1f} MB")

if not checkpoints:
    print("  (none found — training may not have completed)")

Checkpoints found in /proj/ligand_ai/ozery/wdr91_asms:
  last.ckpt                                                     4647.1 MB


In [20]:
# ── Plot training / validation metrics from the Lightning CSV log ──────────────
import glob
import pandas as pd
import matplotlib.pyplot as plt

csv_files = sorted(glob.glob(os.path.join(MODEL_DIR, "**", "metrics.csv"), recursive=True))

if csv_files:
    metrics_df = pd.read_csv(csv_files[-1])
    print(f"Metrics file: {csv_files[-1]}")
    print(f"Columns     : {list(metrics_df.columns)}")
    print()

    # Identify validation accuracy and loss columns
    acc_col  = f"validation.metrics.{TARGET_NAME.lower()}_{ASSAY_TYPE.lower()}_acc"
    loss_col = "validation.losses.total_loss"

    val_metrics = metrics_df.dropna(subset=["epoch"]).copy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    if acc_col in val_metrics.columns:
        val_acc = val_metrics[["epoch", acc_col]].dropna()
        axes[0].plot(val_acc["epoch"], val_acc[acc_col], marker="o", label="val acc")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Accuracy")
        axes[0].set_title(f"Validation Accuracy ({NAME})")
        axes[0].legend()
        axes[0].grid(True)
    else:
        axes[0].text(0.5, 0.5, f"Column not found:\n{acc_col}",
                     ha="center", va="center", transform=axes[0].transAxes)

    if loss_col in val_metrics.columns:
        val_loss = val_metrics[["epoch", loss_col]].dropna()
        axes[1].plot(val_loss["epoch"], val_loss[loss_col], marker="o", color="orange", label="val loss")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].set_title("Validation Loss")
        axes[1].legend()
        axes[1].grid(True)
    else:
        axes[1].text(0.5, 0.5, f"Column not found:\n{loss_col}",
                     ha="center", va="center", transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()

    print("\nFull metrics table (last 10 rows):")
    display(metrics_df.tail(10))
else:
    print("No metrics.csv found — Lightning CSV logger may not have been active.")
    print("Check the trainer logs or ClearML for metrics.")

No metrics.csv found — Lightning CSV logger may not have been active.
Check the trainer logs or ClearML for metrics.
